# Environment Setup

In [ ]:
!uv pip install bitsandbytes xformers triton unsloth vllm==0.10.2
!uv pip install transformers==4.55.4
!uv pip install --no-deps trl==0.22.2

Using Python 3.12.12 environment at: /usr
Audited 5 packages in 112ms
Using Python 3.12.12 environment at: /usr
Audited 1 package in 95ms
Using Python 3.12.12 environment at: /usr
Audited 1 package in 108ms


In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
%cd '/content/drive/MyDrive/multi-reward-math-reasoning/Llama'

/content/drive/MyDrive/multi-reward-math-reasoning/Llama


In [ ]:
%ls

'Llama-3.2-1b outputs'/  'Llama-3.2-3b outputs'/
 Llama-3.2-3B.ipynb       Llama.ipynb


In [ ]:
from unsloth import FastLanguageModel
from trl import SFTConfig, GRPOConfig, SFTTrainer, GRPOTrainer
from vllm import SamplingParams

import gc
import re
import time
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.notebook import tqdm
from datasets import load_dataset, Dataset
from safetensors import safe_open

from peft import LoraConfig, get_peft_model, TaskType
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TextStreamer
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 10-21 23:14:25 [__init__.py:216] Automatically detected platform cuda.
🦥 Unsloth Zoo will now patch everything to make training faster!


# Model Setup

In [ ]:
model_id = 'unsloth/Llama-3.2-1B'          # Select model optimized for instruction-following and reasoning
model_name = model_id.split('/')[-1].lower()  # Extract model name from ID
max_seq_length = 2048                         # Can increase for longer reasoning traces
lora_rank = 32                                # Larger rank = smarter, but slower

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_id,
    max_seq_length=max_seq_length,
    load_in_4bit=False,         # False for LoRA 16bit
    fast_inference=True,        # Enable vLLM fast inference
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.9, # Reduce if out of memory
)
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,                          # Rank: adaptation capacity (16 good for reasoning tasks)
    lora_alpha=lora_rank * 2,             # Scaling factor (typically 2x rank)
    lora_dropout=0.1,                     # Regularization to prevent overfitting
    target_modules=[                      # Remove QKVO if out of memory
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    use_gradient_checkpointing='unsloth', # Reduces memory usage
    random_state=3407,
)

INFO 10-21 23:14:34 [vllm_utils.py:694] Unsloth: Patching vLLM v1 graph capture
INFO 10-21 23:14:34 [vllm_utils.py:722] Unsloth: Patching vLLM v0 graph capture
==((====))==  Unsloth 2025.10.7: Fast Llama patching. Transformers: 4.55.4. vLLM: 0.10.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/Llama-3.2-1B with actual GPU utilization = 88.97%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 39.56 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048. Num Sequences = 320.
Unsloth: vLLM's KV Cache can use up to 32.85 GB. Also swap space = 6 GB.
WARNING 10-21 23:14:43 [compilation.py:45

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 10-21 23:15:05 [default_loader.py:268] Loading weights took 0.73 seconds
INFO 10-21 23:15:05 [punica_selector.py:19] Using PunicaWrapperGPU.
INFO 10-21 23:15:06 [gpu_model_runner.py:2392] Model loading took 2.3782 GiB and 1.934611 seconds
INFO 10-21 23:15:12 [backends.py:539] Using cache directory: /root/.cache/vllm/torch_compile_cache/5df3daed28/rank_0_0/backbone for vLLM's torch.compile
INFO 10-21 23:15:12 [backends.py:550] Dynamo bytecode transform time: 6.09 s
INFO 10-21 23:15:15 [backends.py:161] Directly load the compiled graph(s) for dynamic shape from the cache, took 1.705 s
INFO 10-21 23:15:16 [monitor.py:34] torch.compile takes 6.09 s in total
INFO 10-21 23:15:18 [gpu_worker.py:298] Available KV cache memory: 31.32 GiB
INFO 10-21 23:15:18 [kv_cache_utils.py:864] GPU KV cache size: 1,026,400 tokens
INFO 10-21 23:15:18 [kv_cache_utils.py:868] Maximum concurrency for 2,048 tokens per request: 501.17x
INFO 10-21 23:15:18 [vllm_utils.py:699] Unsloth: Running patched vLLM v1 `

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:04<00:00, 13.42it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 43/43 [00:03<00:00, 13.20it/s]

INFO 10-21 23:15:26 [gpu_model_runner.py:3118] Graph capturing finished in 8 secs, took 0.61 GiB
INFO 10-21 23:15:26 [vllm_utils.py:706] Unsloth: Patched vLLM v1 graph capture finished in 8 secs.


INFO 10-21 23:15:28 [gpu_worker.py:391] Free memory on device (39.03/39.56 GiB) on startup. Desired GPU memory utilization is (0.88969201168322, 35.19 GiB). Actual usage is 2.38 GiB for weight, 1.47 GiB for peak activation, 0.02 GiB for non-torch memory, and 0.61 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=32817368166` to fit into requested memory, or `--kv-cache-memory=36936476160` to fully utilize gpu memory. Current kv cache memory in use is 33633160294 bytes.
INFO 10-21 23:15:28 [core.py:218] init engine (profile, create kv cache, warmup model) took 22.07 seconds
INFO 10-21 23:15:29 [llm.py:295] Supported_tasks: ('generate',)
INFO 10-21 23:15:29 [__init__.py:36] No IOProcessor plugins requested by the model
Unsloth: Just some info: will skip parsing ['q_norm', 'norm2', 'input_layernorm', 'layer_norm2', 'post_attention_layernorm', 'layer_norm1', 'norm1', 'pre_feedforward_layernorm', 'post_feedforward_layernorm', 'post_layernorm', 'k_norm',

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.10.7 patched 16 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


# Chat Template

In [ ]:
# Define structured output format for mathematical reasoning
REASONING_START = '<THINK>'   # Begin reasoning section
REASONING_END = '</THINK>'    # End reasoning section
SOLUTION_START = '<SOLUTION>' # Begin final answer
SOLUTION_END = '</SOLUTION>'  # End final answer

# System prompt that teaches the model our desired reasoning structure
SYSTEM_PROMPT = f'''You are a mathematical reasoning assistant. When given a math problem:
1. Show your step-by-step work between {REASONING_START} and {REASONING_END}.
2. Provide your final numerical answer between {SOLUTION_START} and {SOLUTION_END}.
3. Be precise and show all calculation steps clearly.'''
print(SYSTEM_PROMPT)

You are a mathematical reasoning assistant. When given a math problem:
1. Show your step-by-step work between <THINK> and </THINK>.
2. Provide your final numerical answer between <SOLUTION> and </SOLUTION>.
3. Be precise and show all calculation steps clearly.


In [ ]:
chat_template = ( # Build and assign chat_template to the tokenizer
    # If the very first message is a SYSTEM role, print it + <eos>:
    "{% if messages[0]['role'] == 'system' %}"
      "{{ messages[0]['content'] + eos_token }}"
      "{% set loop_messages = messages[1:] %}"
    "{% else %}"
      # Otherwise, inject our system_prompt + <eos>:
      "{{ '{system_prompt}' + eos_token }}"
      "{% set loop_messages = messages %}"
    "{% endif %}"

    # Now loop over the remaining messages (either user or assistant):
    "{% for message in loop_messages %}"
      "{% if message['role'] == 'user' %}"
        "{{ message['content'] }}"
      "{% elif message['role'] == 'assistant' %}"
        "{{ message['content'] + eos_token }}"
      "{% endif %}"
    "{% endfor %}"

    # If we asked for "add_generation_prompt", append <REASONING> to the end:
    "{% if add_generation_prompt %}{{ '{reasoning_start}' }}"
    "{% endif %}"
)
# Replace with out specific template:
tokenizer.chat_template = chat_template\
    .replace("'{system_prompt}'",   f"'{SYSTEM_PROMPT}'")\
    .replace("'{reasoning_start}'", f"'{REASONING_START}'")

In [ ]:
example_messages = [ # Quick sanity check of the template
    {'role': 'user', 'content': 'Which country has the highest population density?'},
    {'role': 'assistant', 'content': (
        f'{REASONING_START}'
        'I know that country X is small in area but has a huge population, '
        'so its people per square kilometer is extremely high.'
        f'{REASONING_END}{SOLUTION_START}Monaco{SOLUTION_END}'
    )},
    {'role': 'user', 'content': 'Which planet is farthest from the Sun?'},
]
print(tokenizer.apply_chat_template(example_messages, tokenize=False, add_generation_prompt=True))

You are a mathematical reasoning assistant. When given a math problem:
1. Show your step-by-step work between <THINK> and </THINK>.
2. Provide your final numerical answer between <SOLUTION> and </SOLUTION>.
3. Be precise and show all calculation steps clearly.<|end_of_text|>Which country has the highest population density?<THINK>I know that country X is small in area but has a huge population, so its people per square kilometer is extremely high.</THINK><SOLUTION>Monaco</SOLUTION><|end_of_text|>Which planet is farthest from the Sun?<THINK>


# Pre Fine-tuning (SFT)

## Data preparation

In [ ]:
# Use a subset of NVIDIA's Open Math Reasoning dataset, which was filtered to only include high quality DeepSeek R1 traces
sft_dataset = load_dataset('unsloth/OpenMathReasoning-mini', split='cot').to_pandas()
sft_dataset = sft_dataset[['expected_answer', 'problem', 'generated_solution']]

# Try converting to number - if not, replace with NaN
is_number = pd.to_numeric(pd.Series(sft_dataset['expected_answer']), errors='coerce').notnull()
sft_dataset = sft_dataset.iloc[np.where(is_number)[0]] # Select only numbers
sft_dataset

,expected_answer,problem,generated_solution
0,14,Given $\sqrt{x^2+165}-\sqrt{x^2-52}=7$ and $x$...,"<think>\nOkay, let's see. I need to solve the ..."
6,-2,Find the value of the parameter $a$ for which ...,"<think>\nOkay, so I need to find the value of ..."
9,18,What is the sum of all real numbers $x$ for wh...,"<think>\nOkay, so I need to solve the equation..."
13,2,Evaluate the sum \(\sum_{n=1}^\infty \frac{\ph...,"<think>\nOkay, so I need to evaluate the infin..."
17,30,What is the largest positive integer that divi...,"<think>\nAlright, so I need to find the larges..."
...,...,...,...
19243,244,"Let \( p \), \( q \), and \( r \) be the disti...","<think>\nOkay, so I need to find the value of ..."
19245,1,A bug is on the $0$ of a number line. At any p...,"<think>\nOkay, so I have this problem where a ..."
19247,4,A bus left point X for point Y. Two hours late...,"<think>\nOkay, let's tackle this problem step ..."
19248,18,Each interior angle of a regular n-gon measure...,"<think>\nOkay, let's see. I need to find the n..."


In [ ]:
def format_dataset(x): # Format the dataset to follow our GRPO style formatting
    expected_answer = x['expected_answer']
    problem = x['problem']

    # Remove generated <think> and </think>
    thoughts = x['generated_solution'].replace('<think>', '').replace('</think>', '')
    thoughts = thoughts.strip()

    # Add our custom formatting
    final_prompt = REASONING_START + thoughts + REASONING_END + \
                   SOLUTION_START + expected_answer + SOLUTION_END
    return [
        {'role': 'system'   , 'content': SYSTEM_PROMPT},
        {'role': 'user'     , 'content': problem},
        {'role': 'assistant', 'content': final_prompt},
    ]

sft_dataset['messages'] = sft_dataset.apply(format_dataset, axis=1)
print(tokenizer.apply_chat_template(sft_dataset['messages'][0], tokenize=False))

You are a mathematical reasoning assistant. When given a math problem:
1. Show your step-by-step work between <THINK> and </THINK>.
2. Provide your final numerical answer between <SOLUTION> and </SOLUTION>.
3. Be precise and show all calculation steps clearly.<|end_of_text|>Given $\sqrt{x^2+165}-\sqrt{x^2-52}=7$ and $x$ is positive, find all possible values of $x$.<THINK>Okay, let's see. I need to solve the equation √(x² + 165) - √(x² - 52) = 7, and find all positive values of x. Hmm, radicals can be tricky, but maybe if I can eliminate the square roots by squaring both sides. Let me try that.

First, let me write down the equation again to make sure I have it right:

√(x² + 165) - √(x² - 52) = 7.

Okay, so the idea is to isolate one of the radicals and then square both sides. Let me try moving the second radical to the other side:

√(x² + 165) = 7 + √(x² - 52).

Now, if I square both sides, maybe I can get rid of the square roots. Let's do that:

(√(x² + 165))² = (7 + √(x² - 52))².

S

In [ ]:
# Truncate pre fine-tuning sft_dataset to max_seq_length / 2 since we don't want too long reasoning traces
sft_dataset['seq_length'] = sft_dataset['messages'].apply(lambda x: len(tokenizer.apply_chat_template(x)))
print('Token-length percentiles (50/90/99):', np.percentile(sft_dataset['seq_length'], [50, 90, 99]))

threshold = max_seq_length / 2
sft_dataset_filtered = sft_dataset.loc[sft_dataset['seq_length'] <= threshold].copy()
print(f'Remaining for training (<= {threshold} tokens): {len(sft_dataset_filtered)}/{len(sft_dataset)}')

sft_dataset_filtered['text'] = tokenizer.apply_chat_template(sft_dataset_filtered['messages'].values.tolist(), tokenize=False)
sft_dataset_filtered = Dataset.from_pandas(sft_dataset_filtered)
sft_dataset_filtered

Token-length percentiles (50/90/99): [ 3469.   8310.4 14578.9]
Remaining for training (<= 1024.0 tokens): 73/7507


Dataset({
    features: ['expected_answer', 'problem', 'generated_solution', 'messages', 'seq_length', 'text', '__index_level_0__'],
    num_rows: 73
})

## Pre fine-tune to understand custom GRPO formatting

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=sft_dataset_filtered,
    args=SFTConfig(
        dataset_text_field='text',
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        optim='adamw_8bit',
        weight_decay=0.01,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_steps=5,
        logging_steps=5,
        seed=3407,
        report_to='none', # Use this for WandB
    )
)
trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/73 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 73 | Num Epochs = 3 | Total steps = 219
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 1 x 1) = 1
 "-____-"     Trainable parameters = 22,544,384 of 1,258,358,784 (1.79% trained)


Step,Training Loss
5,1.433700
10,1.109500
15,1.001400
20,1.012600
25,0.903300
30,0.943200
35,0.935200
40,0.799900
45,0.873800
50,0.848300


Unsloth: Will smartly offload gradients to save VRAM!


TrainOutput(global_step=219, training_loss=0.5994017287476422, metrics={'train_runtime': 56.6231, 'train_samples_per_second': 3.868, 'train_steps_per_second': 3.868, 'total_flos': 1180872997208064.0, 'train_loss': 0.5994017287476422, 'epoch': 3.0})

## Check if model has learnt to follow the format

In [ ]:
text = tokenizer.apply_chat_template( # Render into a single string and append <REASONING> for generation
    sft_dataset_filtered[1]['messages'][:2],
    tokenize=False, add_generation_prompt=True, # Append the final <REASONING>
)
_ = model.generate(
    **tokenizer(text, return_tensors='pt').to('cuda'),
    temperature=0, max_new_tokens=1024,
    streamer=TextStreamer(tokenizer, skip_prompt=False), # Stream the model's generations (CoT + solution)
)

<|begin_of_text|>You are a mathematical reasoning assistant. When given a math problem:
1. Show your step-by-step work between <THINK> and </THINK>.
2. Provide your final numerical answer between <SOLUTION> and </SOLUTION>.
3. Be precise and show all calculation steps clearly.<|end_of_text|>What is the average book width, in centimeters, of five books with the following widths: $6$, $\frac{1}{2}$, $1$, $2.5$, and $10$?<THINK>Okay, let's see. I need to find the average width of five books. The widths given are 6 cm, 1/2 cm, 1 cm, 2.5 cm, and 10 cm. Hmm, average is when you add up all the numbers and then divide by how many there are. So, first, I need to convert all the numbers to the same format.

Let me look at each number. The ones I know are 6, 1/2, 1, 2.5, and 10. The others are fractions. I can convert the fraction to a decimal by multiplying by 100. So 1/2 is 0.5, 1 is 1, 2.5 is 2.5, and 10 is 10. Wait, but wait. The problem says the widths are given in "cents", which is a differ

In [ ]:
del sft_dataset, sft_dataset_filtered
gc.collect()
torch.cuda.empty_cache()

# Post Fine-tuning (RL)

## Data preparation

In [ ]:
def process_dataset_sample(example): # Convert GSM8K example to conversation format for GRPO training
    return {
        'prompt': [ # Create conversation with system prompt for structured reasoning
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': example['question']},
        ],
        # Extract numerical answer from GSM8K format ('Explanation... #### 42') as Ground truth for reward functions
        'answer': example['answer'].split('####')[1].strip() if '####' in example['answer'] else None
    }

In [ ]:
# train_dataset = load_dataset('openai/gsm8k', 'main', split=['train[:10%]'])
train_dataset = load_dataset('openai/gsm8k', 'main', split='train')
train_dataset = train_dataset.map(process_dataset_sample)

print(f'Training samples: {len(train_dataset):,}\n'
      f"- Sample question: {train_dataset[0]['prompt'][1]['content']}\n"
      f"- Sample answer (ground truth for rewards): {train_dataset[0]['answer']}\n"
      f"- Prompt (system + user):\n{train_dataset[0]['prompt']}")

Training samples: 7,473
- Sample question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
- Sample answer (ground truth for rewards): 72
- Prompt (system + user):
[{'content': 'You are a mathematical reasoning assistant. When given a math problem:\n1. Show your step-by-step work between <THINK> and </THINK>.\n2. Provide your final numerical answer between <SOLUTION> and </SOLUTION>.\n3. Be precise and show all calculation steps clearly.', 'role': 'system'}, {'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'role': 'user'}]


In [ ]:
# Get the top 90% prompt length so we don't accidentally truncate them, i.e. we'll remove the top 10% long prompts
tokenized_dataset = train_dataset.map(
    lambda x: {'tokens': tokenizer.apply_chat_template(x['prompt'], add_generation_prompt=True, tokenize=True)},
    batched=True,
).map(lambda x: {'length': len(x['tokens'])})
print(tokenizer.decode(tokenized_dataset[0]['tokens']))

thresholds = np.percentile(tokenized_dataset['length'], [50, 90, 99])
max_prompt_length = int(thresholds[1])
print('Token-length percentiles (50/90/99):', thresholds, '=> Choose max_prompt_length =', max_prompt_length)

# Filter only samples smaller than 90% max length
train_dataset = train_dataset.select(np.where(np.array(tokenized_dataset['length']) <= max_prompt_length)[0])
print(f'Remaining for training (<= {max_prompt_length} tokens): {len(train_dataset)}/{len(tokenized_dataset)}')
del tokenized_dataset

You are a mathematical reasoning assistant. When given a math problem:
1. Show your step-by-step work between <THINK> and </THINK>.
2. Provide your final numerical answer between <SOLUTION> and </SOLUTION>.
3. Be precise and show all calculation steps clearly.<|end_of_text|>Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<THINK>
Token-length percentiles (50/90/99): [116. 149. 187.] => Choose max_prompt_length = 149
Remaining for training (<= 149 tokens): 6749/7473


## Regex Patterns

In [ ]:
# Match the reasoning sections and answers
match_format = re.compile(
    # rf'^[\s]{{0,}}'                                     # Optional whitespace at start
    # rf'{REASONING_START}.+?{REASONING_END}.*?'          # Reasoning section (non-greedy)
    rf'{REASONING_END}.*?'                              # We always prepend REASONING_START
    rf'{SOLUTION_START}(.+?){SOLUTION_END}'             # Solution section with capture group
    rf'[\s]{{0,}}(?:{re.escape(tokenizer.eos_token)})?' # Add optional EOS token matching
    rf'[\s]{{0,}}$',                                    # Optional whitespace at end
    flags=re.MULTILINE | re.DOTALL,                     # Multi-line matching with . matching newlines
)
match_format.findall( # Verify it works
    f'{REASONING_START}Let me think!{REASONING_END}'\
    f'{SOLUTION_START}\n2\n{SOLUTION_END}\n\n',
)

['\n2\n']

In [ ]:
# Sometimes it might not be 1 number as the answer, but like a sentence.
# For example: 'The solution is $20' -> we extract 20
# We also remove possible commas for example as in 123,456
match_number = re.compile(
    rf'{SOLUTION_START}.*?[\s]{{0,}}([-]?[\d\.\,]{{1,}})', # Extract numbers from solution section
    flags=re.MULTILINE | re.DOTALL | re.IGNORECASE,        # Flexible pattern matching
)
print(match_number.findall('<SOLUTION>  0.34  </SOLUTION>'))
print(match_number.findall('<SOLUTION>  123,456  </SOLUTION>'))
print(match_number.findall('<SOLUTION>  -0.234  </SOLUTION>'))
print(match_number.findall('<SOLUTION>17</SOLUTION>'))

['0.34']
['123,456']
['-0.234']
['17']


## Multi-reward design

In [ ]:
def match_format_strictly(completions, **kwargs) -> list[float]:
    ''' Reward Function 1: Exact Format Compliance
    High reward (3.0) for perfect format adherence
    Ensures model learns the complete structured output pattern
    '''
    return [
        3.0 if match_format.search(completion[0]['content']) else 0.0
        for completion in completions
    ]

In [ ]:
# If it fails, reward the model if it at least follows the format partially, by counting each symbol
def match_format_softly(completions, **kwargs) -> list[float]:
    ''' Reward Function 2: Partial Format Credit
    Graduated scoring for format elements
    Encourages learning individual components even if not perfect
    '''
    rewards = []
    for completion in completions:
        reward = 0
        response = completion[0]['content']

        # Count how many keywords are seen - we penalize if too many!
        # Award +0.5 for correct token count, -0.5 for wrong count
        # No need to reward REASONING_START since we always prepend it!
        # reward += 0.5 if response.count(REASONING_START) == 1 else -0.5
        reward += 0.5 if response.count(REASONING_END) == 1 else -0.5
        reward += 0.5 if response.count(SOLUTION_START) == 1 else -0.5
        reward += 0.5 if response.count(SOLUTION_END) == 1 else -0.5
        rewards.append(reward)
    return rewards

In [ ]:
# Extract the generated answer, and reward or penalize it
def check_answer_correctness(completions, answer, **kwargs) -> list[float]:
    ''' Reward Function 3: Graduated scoring for mathematical accuracy
    - 5.0: Exact string match gets full points
    - 2.0: Within 10% (close answer)
    - 1.5: Within 20% (reasonable attempt)
    - -2.5: Wrong answer (penalty for incorrect math)
    '''
    responses = [completion[0]['content'] for completion in completions]
    extracted_responses = [ # Extract answers using format pattern
        guess.group(1) if (guess := match_format.search(r)) else None
        for r in responses
    ]
    rewards = []
    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None: # No extractable answer
            rewards.append(-2.0)
            continue

        if guess == true_answer: rewards.append(5.0)                   # Correct answer gets 5 points!
        elif guess.strip() == true_answer.strip(): rewards.append(3.5) # Match if spaces are seen, but less reward
        else: # Try numerical comparison for partial credit
            try: # We also reward it based on how close the answer is to the true one via ratios
                ratio = float(guess) / float(true_answer)     # If the answer is within some range, reward it!
                if 0.9 <= ratio <= 1.1: rewards.append(2.0)   # Within 10%
                elif 0.8 <= ratio <= 1.2: rewards.append(1.5) # Within 20%
                else: rewards.append(-2.5)                    # Penalize wrong answers
            except (ValueError, ZeroDivisionError):
                rewards.append(-4.5)                          # Invalid numerical format
    return rewards

In [ ]:
def check_numbers_extraction(prompts, completions, answer, **kwargs) -> list[float]:
    ''' Reward Function 4: Number Extraction Ability
    Tests the model's ability to extract numerical values from solution sections
    Complementary to exact format matching - focuses on parsing capability
    '''
    question = prompts[0][-1]['content'] # Exclude system prompt
    responses = [completion[0]['content'] for completion in completions]

    extracted_responses = [ # Extract numbers from solution sections using number pattern
        guess.group(1) if (guess := match_number.search(r)) else None
        for r in responses
    ]
    rewards = []

    # Print only every few steps
    check_numbers_extraction.counter = getattr(check_numbers_extraction, 'counter', 0) + 1
    if check_numbers_extraction.counter % 100 == 0:
        print(
            '==' * 100,
            f'\nQuestion: {question}'
            f'\nPrediction: {extracted_responses[0]}, GT Answer: {answer[0]}'
            f'\nResponse:\n{responses[0]}'
        )
    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None: # No extractable number
            rewards.append(-2.5)
            continue

        try: # Simple numerical equality check
            true_val = float(true_answer.strip())             # Convert to numbers
            guess_val = float(guess.strip().replace(',', '')) # Remove commas like in 123,456
            rewards.append(3.5 if guess_val == true_val else -1.5)
        except (ValueError, TypeError):
            rewards.append(0) # Invalid number format
    return rewards

## GRPO training setup

In [ ]:
max_prompt_length = 149 + 1
max_completion_length = max_seq_length - max_prompt_length
vllm_sampling_params = SamplingParams(
    min_p = 0.1,
    top_p = 1.0,
    top_k = -1,
    stop = [tokenizer.eos_token],
    include_stop_str_in_output = True,
)

In [ ]:
training_args = GRPOConfig(          # Configure GRPO training parameters for mathematical reasoning
    output_dir=f'/tmp/{model_name}', # Directory for checkpoints and logs
    vllm_sampling_params=vllm_sampling_params,
    # Training speed control
    num_train_epochs=1,              # Total number of training epochs
    per_device_train_batch_size=2,   # Small batch for GPU memory constraints
    gradient_accumulation_steps=8,   # Effective batch size = 2 * 8 = 16
    # Computing the loss: https://huggingface.co/docs/trl/main/grpo_trainer#computing-the-loss
    scale_rewards='batch',           # Calculate mean at local/group level and std at global/batch level enables more robust reward shaping
    loss_type='dr_grpo',             # Fully remove response length bias, dividing by a constant instead of the sequence length
    # Precision & Optimization
    optim='adamw_8bit',              # adamw_torch_fused, adamw_8bit, paged_adamw_8bit
    weight_decay=0.1,                # Regularization
    max_grad_norm=0.1,               # Aggressive gradient clipping for stable training
    gradient_checkpointing=True,
    bf16=torch.cuda.is_available(),  # Enable mixed-precision training if a CUDA GPU is available (faster, less memory)
    # Learning rate scheduling
    learning_rate=1e-5,              # Conservative LR to prevent destabilizing reasoning
    warmup_ratio=0.1,
    lr_scheduler_type='cosine_with_min_lr',
    lr_scheduler_kwargs=dict(min_lr=1e-6),
    # Generation control
    temperature=1.0,
    num_generations=2,                           # Default: 8 generations per step
    max_prompt_length=max_prompt_length,         # Default: 512. Sufficient for complex word problems
    max_completion_length=max_completion_length, # Default: 256. Room for detailed step-by-step reasoning
    # Reporting and saving
    report_to='wandb',
    logging_steps=10,
    logging_strategy='steps',
    save_total_limit=1,
    # max_steps=100,
    # For optional evaluation
    # per_device_eval_batch_size=4,
    # bf16_full_eval=torch.cuda.is_available(),
    # eval_strategy='steps',                       # Evaluate after each epoch
    # load_best_model_at_end=True,                 # Load the best model based on validation loss
)

## Train the model

In [ ]:
%%time
trainer = GRPOTrainer(            # Initialize GRPO trainer with multi-reward system
    model=model,                  # LoRA-adapted quantized model
    processing_class=tokenizer,
    train_dataset=train_dataset,  # Processed GSM8K dataset
    args=training_args,           # Training configuration
    reward_funcs=[                # 4 complementary reward functions
        match_format_strictly,    # Perfect structure compliance
        match_format_softly,      # Partial format credit
        check_answer_correctness, # Mathematical accuracy
        check_numbers_extraction, # Number parsing ability
    ]
)
trainer.train()
trainer.save_model(f'./{model_name}_grpo')

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,749 | Num Epochs = 1 | Total steps = 843
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 22,544,384 of 1,258,358,784 (1.79% trained)
wandb: Currently logged in as: walteryeyint (walteryeyint-university-of-technology-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / match_format_strictly / mean,rewards / match_format_strictly / std,rewards / match_format_softly / mean,rewards / match_format_softly / std,rewards / check_answer_correctness / mean,rewards / check_answer_correctness / std,rewards / check_numbers_extraction / mean,rewards / check_numbers_extraction / std
10,0.477100,-1.009375,4.822995,1490.768750,806.100000,1898.000000,0.368750,1244.316138,806.100000,1667.800000,0.595338,1.837500,1.465959,0.350000,1.461040,-1.596875,1.749435,-1.600000,1.197938
20,0.456000,-1.440625,4.378381,1468.968750,702.700000,1898.000000,0.381250,1208.576660,702.700000,1760.700000,0.603488,1.762500,1.481613,0.287500,1.454915,-1.856250,1.485987,-1.634375,1.152680
30,0.440500,-0.790625,4.802660,1440.943750,779.800000,1898.000000,0.350000,1198.525366,779.800000,1726.900000,0.750126,1.893750,1.445388,0.431250,1.414672,-1.596875,1.745921,-1.518750,1.270233
40,0.540500,-1.537500,3.594696,1485.900000,724.300000,1898.000000,0.356250,1252.638379,724.300000,1749.100000,0.565864,1.893750,1.477334,0.431250,1.451251,-2.084375,0.952263,-1.778125,0.691532
50,0.466400,-0.968750,3.574081,1410.643750,819.100000,1898.000000,0.281250,1213.729956,819.100000,1778.400000,0.617871,2.118750,1.323523,0.612500,1.317550,-2.043750,1.296092,-1.656250,0.724773
60,0.556300,-0.128125,3.885747,1337.612500,753.600000,1898.000000,0.206250,1193.396729,753.600000,1704.800000,0.556046,2.381250,1.234781,0.881250,1.234781,-1.871875,1.642106,-1.518750,0.932872
70,0.390300,0.046875,3.015435,1208.318750,618.600000,1880.800000,0.131250,1104.777173,618.600000,1751.200000,0.582990,2.568750,0.962815,1.056250,1.001537,-2.037500,1.379420,-1.540625,0.638094
80,0.133300,0.550000,3.551148,1224.618750,606.400000,1855.300000,0.131250,1122.345264,606.400000,1708.500000,0.683869,2.568750,0.962815,1.081250,0.944959,-1.681250,1.694006,-1.418750,0.841730
90,0.396500,0.596875,2.863108,1086.687500,626.800000,1794.200000,0.068750,1027.683960,626.800000,1588.600000,0.634889,2.756250,0.719337,1.256250,0.719337,-2.009375,1.453465,-1.406250,0.698940
100,0.052500,1.640625,3.105896,1001.787500,579.300000,1628.300000,0.006250,995.988336,579.300000,1598.400000,0.672996,2.962500,0.150000,1.456250,0.175000,-1.640625,2.078297,-1.137500,0.995239


Question: The tallest building in the world is 100 feet tall.  If the second tallest is half that tall, and the third tallest is half as tall as the second, and the fourth is one-fifth as tall as the third, how tall are all 4 buildings put together?
Prediction: 177, GT Answer: 180
Response:
Okay, let's see. I need to figure out how tall 4 buildings are when they’re connected, where the tallest is 100 feet tall and the others are half that. Hmm, let me break this down.

First, the second building needs to be half the size of the first, the third half of the second, and the fourth one fifth the size of the third. It looks like there are three distinct segments here. So the tallest building is called S, the second is S/2, the third S/3, and the fourth S/4.

I remember that formula for adding up fractions... It should be (A + B)/2, right? Because adding two fractions like A/B and (A+B)/2. So in this case, S + S/2 + S/3 + S/4.

Wait, does S equal 100? Because the problem says the tallest is

# Evaluation

## Resource usage

In [ ]:
# Memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f'GPU = {gpu_stats.name}. Max memory = {max_memory} GB.')
print(f'{start_gpu_memory} GB of memory reserved.')

GPU = NVIDIA A100-SXM4-40GB. Max memory = 39.557 GB.
38.471 GB of memory reserved.


In [ ]:
# Extract runtime info
last_log = trainer.state.log_history[-1] # Final memory and time stats
train_seconds = last_log['train_runtime']
samples_per_second = last_log.get('train_samples_per_second', None)

# Recompute GPU memory stats
used_memory   = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
used_for_lora = round(used_memory - start_gpu_memory, 2)
used_pct      = round(used_memory / max_memory * 100, 2)
lora_pct      = round(used_for_lora / max_memory * 100, 2)

print(f'Training time: {train_seconds:.1f} seconds ({train_seconds / 60:.2f} minutes)')
if samples_per_second: print(f'Throughput: {samples_per_second:.1f} samples/second')
print(f'Peak VRAM usage: {used_memory} GB ({used_pct}% of max memory)')
print(f'VRAM for training: {used_for_lora} GB ({lora_pct}% of max memory)')

Training time: 8710.1 seconds (145.17 minutes)
Throughput: 0.8 samples/second
Peak VRAM usage: 38.47 GB (97.25% of max memory)
VRAM for training: -0.0 GB (-0.0% of max memory)


## Verify LoRA is actually trained

In [ ]:
example_text = 'What is the sqrt of 101?'
# example_text = 'Solve (x + 2)^2 = 0'
# example_text = "How many r's are in strawberry?"

sampling_params = SamplingParams(
    temperature=1.0,
    top_k=50,
    max_tokens=max_completion_length,
)
print(model.fast_generate( # Try the model without any GRPO trained
    example_text, sampling_params=sampling_params,
    lora_request=None
)[0].outputs[0].text)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

 What does that mean? I used to take trig and was just able to memorize all the formulas for basic square roots, cube roots and what not. I could even graph it on some graphing program. But then one day I was walking around with a pencil and a compass and I was thinking that maybe I had a much bigger picture. But now I'm scared. I don't know what questions I should ask, or whether I should just say screw it and learn something new.
I could also use some help with the basics of trig. I know about sine, cosine, tangent . . . but what about those little circles I was just starting to get a basic idea about?
Welcome. In order to help you, I can guess at what you're asking.
What is the sqrt of 101? It is not a simple answer.
In order for something to help me, I'd have to know a fair bit about you. What do you want to learn next. You ask what kind of things I'm comfortable with . . .
I've been learning about trigonometry for five years. At one point, I was pretty good, and thought I could ju

In [ ]:
tensors = {}
with safe_open(f'./{model_name}_grpo/adapter_model.safetensors', framework='pt') as f:
    for key in f.keys(): # Verify both A and B are non zero
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert(n_zeros.item() != tensor.numel())

In [ ]:
# Load the LoRA and test without using system prompt
# which should not (or minimal) affect the model's original reasoning ability
text = tokenizer.apply_chat_template(
    [{'role': 'user', 'content': example_text}],
    add_generation_prompt=True, tokenize=False,
)
print(model.fast_generate(
    text, sampling_params=sampling_params,
    lora_request=model.load_lora(f'./{model_name}_grpo'),
)[0].outputs[0].text)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Okay, let's see. The problem is: What is the sqrt of 101? First, right, I need to figure out the square root of 101. So what should that be? Hmm. Well, since I know about the Pythagorean theorem, maybe I can use that here. The square root of a number x squared equals x. So 101 squared is 101, right? So the square root should be 101. Got it. So the answer should be 11. 

Wait, the question is about the sqrt of 101. Maybe the problem could've worded differently? Like, maybe they want to know what x is when y=sqrt(101). So that's confusing. Because sqrt of 101 is 11. But y is 11. And therefore, the square root is 11. So that makes sense. Because if you plug y=11 into the equation to find x, then x=11. So yes, the answer should be 11. That all makes sense. So I think that's it. I don't see any other possible answers here. Hence, the answer should be 11.
To determine the square root of 101, we can use the Pythagorean theorem:
$$ x^2 = 101 \Rightarrow x = \sqrt{101} $$
Thus, the square root 

In [ ]:
# Test using system prompt
text = tokenizer.apply_chat_template([
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user'  , 'content': example_text},
], add_generation_prompt=True, tokenize=False)

# Compare results with system prompt but without LoRA
print(model.fast_generate(
    text, sampling_params=sampling_params,
    lora_request=None,
)[0].outputs[0].text)

# Reasoning model is much better - it's not always correct, since we only trained it for an hour
# It'll be better if we extend the sequence length and train for longer
print(model.fast_generate(
    text, sampling_params=sampling_params,
    lora_request=model.load_lora(f'./{model_name}_grpo'),
)[0].outputs[0].text)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

 2 - 23 100% SOLUTION </SOLUTION> 3 - 2 = 1 8
<THINK> 2 - 23 100% SOLUTION </SOLUTION> 3 = 1 5
2 - 23 100% SOLUTION </SOLUTION> 3+2 = 5 5
3 - 2 = 1 8


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Okay, let's try to figure out the square root of 101. Hmm, I remember that if I multiply a number by 10 and then take the square root, that gives me the original number. That's probably where I'm supposed to start. Let me try that.

Oh, let me check: 101 times 10 equals 1000, and then I need to take the square root of that. So maybe...70? No, the square root of 1000 should be just 70. That seems right. So the answer should be 70.

Wait, but let me verify. If I square root 1000, does it equal 1000? Because sqrt(1000) is 70, right? So the answer should be 70. And yeah, that seems correct. So I think that's right.

But let me make sure I didn't skip any steps. I triple-checked. If I enter 101 and then select 10 from the numeric keypad, the result should be 70. So that's correct. So the answer is probably 70.

I don't think there's another method. I remember that's the standard approach. If I need to square a number, multiply it by 10. Then, when taking the square root, 10 goes away, and t

## Performance on Test set

In [ ]:
# test_dataset = load_dataset('openai/gsm8k', 'main', split=['test[:10%]'])
test_dataset = load_dataset('openai/gsm8k', 'main', split='test').map(process_dataset_sample)
test_texts = [
    tokenizer.apply_chat_template([
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': sample['prompt'][1]['content']},
    ], add_generation_prompt=True, tokenize=False)
for sample in test_dataset]
print(f'Testing samples:', len(test_dataset))

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

Testing samples: 1319


In [ ]:
outputs_with_lora = model.fast_generate(
    test_texts, sampling_params=sampling_params,
    lora_request=model.load_lora(f'./{model_name}_grpo'),
)
outputs_without_lora = model.fast_generate(
    test_texts, sampling_params=sampling_params,
    lora_request=None,
)

Adding requests:   0%|          | 0/1319 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1319 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

Adding requests:   0%|          | 0/1319 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1319 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

In [ ]:
# Compare the correct amount of using and not using LoRA
no_lora_correct_format_cnt = lora_correct_format_cnt = 0
no_lora_correct_answer_cnt = lora_correct_answer_cnt = 0
no_lora_correct_all_cnt = lora_correct_all_cnt = 0
num_test_samples = len(test_dataset)

for output_with_lora, output_without_lora, answer in zip(outputs_with_lora, outputs_without_lora, test_dataset['answer']):
    correct_format = match_format.search(output_with_lora.outputs[0].text)
    correct_answer = (guess := match_number.search(output_with_lora.outputs[0].text)) and guess.group(1) == answer
    correct_all = correct_format and correct_answer
    if correct_format: lora_correct_format_cnt += 1
    if correct_answer: lora_correct_answer_cnt += 1
    if correct_all: lora_correct_all_cnt += 1

    correct_format = match_format.search(output_without_lora.outputs[0].text)
    correct_answer = (guess := match_number.search(output_without_lora.outputs[0].text)) and guess.group(1) == answer
    correct_all = correct_format and correct_answer
    if correct_format: no_lora_correct_format_cnt += 1
    if correct_answer: no_lora_correct_answer_cnt += 1
    if correct_all: no_lora_correct_all_cnt += 1

pd.DataFrame({
    'Without LoRA': {
        'Correct Format': f'{no_lora_correct_format_cnt}/{num_test_samples} ({no_lora_correct_format_cnt / num_test_samples * 100:.2f}%)',
        'Correct Answer': f'{no_lora_correct_answer_cnt}/{num_test_samples} ({no_lora_correct_answer_cnt / num_test_samples * 100:.2f}%)',
        'Correct Both': f'{no_lora_correct_all_cnt}/{num_test_samples} ({no_lora_correct_all_cnt / num_test_samples * 100:.2f}%)',
    },
    'With LoRA': {
        'Correct Format': f'{lora_correct_format_cnt}/{num_test_samples} ({lora_correct_format_cnt / num_test_samples * 100:.2f}%)',
        'Correct Answer': f'{lora_correct_answer_cnt}/{num_test_samples} ({lora_correct_answer_cnt / num_test_samples * 100:.2f}%)',
        'Correct Both': f'{lora_correct_all_cnt}/{num_test_samples} ({lora_correct_all_cnt / num_test_samples * 100:.2f}%)',
    },
    'Improvement': {
        'Correct Format': f'+{lora_correct_format_cnt - no_lora_correct_format_cnt} ({(lora_correct_format_cnt - no_lora_correct_format_cnt) / num_test_samples * 100:.2f}%)',
        'Correct Answer': f'+{lora_correct_answer_cnt - no_lora_correct_answer_cnt} ({(lora_correct_answer_cnt - no_lora_correct_answer_cnt) / num_test_samples * 100:.2f}%)',
        'Correct Both': f'+{lora_correct_all_cnt - no_lora_correct_all_cnt} ({(lora_correct_all_cnt - no_lora_correct_all_cnt) / num_test_samples * 100:.2f}%)',
    }
}).T

,Correct Format,Correct Answer,Correct Both
Without LoRA,99/1319 (7.51%),12/1319 (0.91%),0/1319 (0.00%)
With LoRA,1273/1319 (96.51%),66/1319 (5.00%),66/1319 (5.00%)
Improvement,+1174 (89.01%),+54 (4.09%),+66 (5.00%)


# Inference

In [ ]:
def generate_with_reasoning(questions, max_completion_length=512):
    conversations = [[                        # Format input using conversation template
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': question},
    ] for question in questions]

    prompts = [tokenizer.apply_chat_template( # Apply chat template and tokenize
        conversation,
        add_generation_prompt=True,         # Add assistant prompt
        tokenize=False,                     # Return string, not tokens
    ) for conversation in conversations]

    # Generate response with reasoning-optimized parameters
    inputs = tokenizer(prompts, return_tensors='pt', padding=True).to(model.device)
    start_time = time.time()
    with torch.no_grad():
        output_ids = model.generate(           # Generate response with reasoning-optimized parameters
            **inputs,
            max_new_tokens=max_completion_length,
            temperature=0.7,                # Balance creativity and consistency
            top_p=0.9,                      # Nucleus sampling for quality
            do_sample=True,                 # Enable sampling for varied reasoning paths
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,         # Reduce repetitive reasoning steps
            length_penalty=1.0,             # Neutral preference for response length
            early_stopping=True,            # Stop at natural completion
            streamer=TextStreamer(tokenizer, skip_prompt=True),
        )
    end_time = time.time()
    inference_duration = end_time - start_time
    num_generated_tokens = output_ids.shape[1] - inputs['input_ids'].shape[1]

    output_ids = output_ids[:, inputs['input_ids'][0].shape[-1]:output_ids.shape[-1]]
    responses = tokenizer.batch_decode(output_ids, skip_special_tokens=True) # Decode and extract only the generated portion
    return responses, inference_duration, num_generated_tokens

In [ ]:
test_dataset = load_dataset('openai/gsm8k', 'main', split='test').map(process_dataset_sample)
gsm8k_question = test_dataset[0]['question']
expected_answer = test_dataset[0]['answer']

print('Question:', gsm8k_question, '\nResponse:')
gsm8k_responses, inference_duration, num_generated_tokens = generate_with_reasoning([gsm8k_question], max_completion_length)
gsm8k_response = gsm8k_responses[0]
print('Inference time (secs):', inference_duration)
print('Generated tokens:', num_generated_tokens)

Question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market? 
Response:
Okay, let's see here. So Janet has three chickens that lay 16 eggs a day. Every morning when she gets up, she eats one of them for breakfast and makes some muffins with the other two. Then she sells what's left over at the farmer's market for $2 each. The question is asking how much money she makes every day.

Alright, so I need to set up this equation here: total revenue (sales price) minus cost of production equals profit. Let me break it down.

Revenue = 16 eggs * $2 per egg = $32

Cost of production would be the number of eggs eaten by Janet and baked into muffins, which we know as 4*3=12. Therefore, 32 - 12 = 20. So the amount she makes is 20 dollars per day.

Wait, but hold on. There

In [ ]:
# Validate format compliance
has_solution = SOLUTION_START in gsm8k_response and SOLUTION_END in gsm8k_response
print('Reasoning section:', REASONING_END in gsm8k_response)
print('Solution section:', has_solution)

if has_solution: # Check answer accuracy if solution section exists
    try:
        solution_text = gsm8k_response.split(SOLUTION_START)[1].split(SOLUTION_END)[0].strip()
        extracted_number = ''.join(filter(str.isdigit, solution_text))
        expected_number = ''.join(filter(str.isdigit, expected_answer))
        print('Extracted:', solution_text)
        print('Expected:', expected_answer)
        print('Correct:', extracted_number == expected_number)
    except:
        print('Could not extract solution')

Reasoning section: True
Solution section: True
Extracted: 20
Expected: 18
Correct: False
